# पाठ १३ - एजेन्ट स्मृति


## सेटअप

यस नोटबुकले देखाउँछ कि कसरी **Microsoft Agent Framework** (MAF) प्रयोग गरेर **दिर्घकालीन स्मृति** सहितको यात्रा बुकिङ एजेन्ट बनाउन सकिन्छ।

तपाईंले सिक्नुहुनेछ कि विभिन्न प्रकारका एजेन्ट स्मृति — कार्यशील, छोटो अवधिको, र दिर्घकालीन — कसरी एक एजेन्टले कुराकानीहरूमा जानकारी कसरी राख्छ र प्रयोग गर्छ भनी प्रभाव पार्छ।

**पूर्व आवश्यकताहरू:**
- एउट Azure AI Foundry परियोजना जसमा एक डिप्लोय गरिएको च्याट मोडेल छ (जस्तै `gpt-4o-mini`)।
- Azure CLI सँग लगइन गरिएको छ — तपाईँको टर्मिनलमा `az login` चलाउनुहोस्।
- `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको Azure AI Foundry परियोजना अन्त्यबिन्दु।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईँले डिप्लोय गरेको मोडेलको नाम।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजेन्ट मेमोरीका प्रकारहरू

एआई एजेन्टहरूले विभिन्न प्रकारका मेमोरीहरूको उपयोग गर्न सक्छन्, जसले प्रत्येकले फरक उद्देश्य पूरा गर्छ:

### वर्किङ मेमोरी
संवाद थ्रेड आफैँ — एउटै सत्रमा आदानप्रदान गरिएका सन्देशहरू। एजेन्टले coherence कायम राख्नका लागि सोही थ्रेडका पहिलाका सन्देशहरूमा फिर्ता हेर्न सक्छ। MAF मा यो **`agent.create_session()`** मार्फत सिर्जना हुन्छ, जुन `AgentSession` फिर्ता दिन्छ।

### छोटो-कालीन मेमोरी
केही कार्य वा सत्रको अवधिका लागि कायम रहने जानकारी तर स्थायी रूपमा संग्रहित नहुने। उदाहरणका लागि, एजेन्टले बहु-पटकको योजना बनाउने संवादको क्रममा तथ्यहरू संकलन गर्न सक्छ र अन्तिम यात्रा तालिका उत्पादन गर्न तिनीहरू प्रयोग गर्न सक्छ।

### दीर्घकालीन मेमोरी
पसन्द र तथ्यहरू जुन **सत्रहरूमा पनि कायम रहन्छन्**। फर्कने प्रयोगकर्ताले आफ्नो आहार प्रतिबन्धहरू वा यात्रा शैली फेरि दोहोर्याउनु पर्दैन। दीर्घकालीन मेमोरी प्रायः बाह्य स्टोरद्वारा समर्थित हुन्छ — डाटाबेस, फाइल, वा भेक्टर इन्डेक्स — र उपकरणहरू मार्फत एजेन्टसम्म ल्याइन्छ।


## सत्रहरूसँग कार्यस्मृति

स्मृतिको सबैभन्दा सरल रूप हो वार्तालाप सत्र। जब तपाईं सोही सत्र (जुन `agent.create_session()` मार्फत सिर्जना गरिएको हो) निरन्तर `agent.run()` कलहरूमा पास गर्नुहुन्छ, एजेन्टले सो वार्तालापको पूर्ण इतिहास देख्छ र अघिल्लो विवरणहरू सम्झन सक्छ।

आउनुहोस्, एउटा यात्रा एजेन्ट बनाउँ र कार्यस्मृति देखाऔं।


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजेन्टले बजेट सहीसँग सम्झियो किनभने दुबै सन्देशनहरूले एउटै सत्र साझा गर्छन्। यो **कार्य स्मृति** हो — यो सत्रको जीवनकालका लागि मात्र अस्तित्वमा हुन्छ।

### नयाँ थ्रेडसँग के हुन्छ?

यदि हामी **नयाँ** सत्र सिर्जना गर्छौं भने, एजेन्टसँग अघिल्लो वार्तालापको कुनै पनि स्मृति हुँदैन:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## दीर्घकालीन सम्झना ढाँचा

प्रयोगकर्ता प्राथमिकताहरू **सेसनहरूमा** सम्झनका लागि, हामीलाई कुराकानी थ्रेड बाहिर रहने स्थायी भण्डार आवश्यक हुन्छ। एजेन्टले यो भण्डारलाई **उपकरणहरू** मार्फत पहुँच गर्छ — फंक्शनहरू जसलाई यसले जानकारी बचत र पुनः प्राप्त गर्न कल गर्न सक्छ।

तल हामीले एउटा सरल इन-मेमोरी प्राथमिकता भण्डार लागू गरेका छौं (उत्पादनमा तपाईंले यसलाई डेटाबेस वा भेक्टर सूचकांकसँग समर्थन गर्नुहुनेछ) र यसलाई एजेन्टले प्रयोग गर्न सक्ने उपकरणहरूका रूपमा एक्स्पोज गरेका छौं।

### वास्तुकला
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### परिदृश्य १ — पहिलो पटकको प्रयोगकर्ताले वार्षिकी भ्रमण बुक गर्छ

सारा पहिलो पटक भ्रमण गर्छिन्। एजेन्टले उपकरणहरूको माध्यमबाट उनका प्राथमिकताहरू सुरक्षित गर्नुपर्छ र तिनीहरूलाई होटलहरू सिफारिश गर्न प्रयोग गर्नुपर्छ।


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### परिदृश्य २ — साराह हप्तौंपछि फर्किन्छिन्

साराहले **एक नयाँ थ्रेड** सुरु गर्छिन् (नयाँ सत्रको नक्कल गर्दै)। कार्य स्मृति खाली छ, तर दीर्घकालीन प्राथमिकता भण्डारमा अझै उनको जानकारी छ। एजेन्टले यसलाई पुनः प्राप्त गर्नुपर्छ र सुझावहरूलाई वैयक्तिकृत गर्न यसको प्रयोग गर्नुपर्छ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

यस पाठमा तपाइँले तीन प्रकारका एजेन्ट मेमोरीहरू सिक्नुभयो र तिनीहरूलाई Microsoft Agent Framework सँग कसरी कार्यान्वयन गर्ने:

| मेमोरी प्रकार | MAF मेकानिज्म | जीवनकाल |
|---|---|---|
| **काम गर्ने** | `agent.create_session()` | एकल कुराकानी |
| **छोटो अवधिको** | थ्रेड भित्र संचय गरिएको सन्दर्भ | एकल कार्य / सत्र |
| **दीर्घकालिन** | `@tool` कार्यहरू मार्फत पहुँच गरिएको बाह्य भण्डारण | सत्रहरूमा |

### मुख्य बुँदाहरू
1. **`agent.create_session()`** ले काम गर्ने मेमोरी प्रदान गर्दछ — एजेन्टले सत्र भित्र सम्पूर्ण कुराकानी इतिहास देख्छ।
2. **नयाँ सत्रहरूले सन्दर्भ हराउँछन्** — दीर्घकालिन मेमोरी बिना एजेन्टले विगतका कुराकानीहरू सम्झन सक्दैन।
3. **`@tool` कार्यहरूले पुलको काम गर्छन्** — तिनीहरूले एजेन्टलाई एक निरन्तर भण्डारणबाट जानकारी बचत र पुनःप्राप्त गर्न अनुमति दिन्छन्।
4. **व्यक्तिगतरण समयसँग सुधार हुन्छ** — जति बढी प्राथमिकताहरू संग्रहित हुन्छन्, एजेन्टका सिफारिसहरू उत्तम हुन्छन्।

### वास्तविक-विश्व प्रयोगहरू
- **ग्राहक सेवा**: ग्राहक इतिहास र प्राथमिकताहरू सम्झनुहोस्
- **व्यक्तिगत सहायकहरू**: दिन वा हप्ताहरूमा सन्दर्भ कायम राख्नुहोस्
- **स्वास्थ्य सेवा**: रोगी जानकारी र प्राथमिकताहरू ट्र्याक गर्नुहोस्
- **इ-वाणिज्य**: इतिहासमा आधारित व्यक्तिगत खरिद

### अर्को चरणहरू
- इन-मेमोरी dict लाई डेटाबेस वा भेक्टर स्टोर (जस्तै Azure AI Search) सँग प्रतिस्थापन गर्नुहोस्
- समय-सम्वेदनशील जानकारीका लागि मेमोरी समाप्ति थप्नुहोस्
- साझा मेमोरी भएका बहु-एजेन्ट प्रणालीहरू निर्माण गर्नुहोस्
- ज्ञान-ग्राफ-सम्बन्धित मेमोरीका लागि Cognee नोटबुक अन्वेषण गर्नुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकृति**:  
यो दस्तावेज कृत्रिम बुद्धिमत्ता अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) द्वारा अनुवाद गरिएको हो। हामी शुद्धताको लागि प्रयासरत छौं, तर कृत्रिम अनुवादमा त्रुटि वा असम्बद्धता हुन सक्नेछ भने कृपया सचेत रहनुहोस्। मूल दस्तावेज यसको स्वदेशी भाषामा अधिकारिक स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीका लागि पेशेवर मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट हुने कुनै पनि गलतफहमी वा गलत व्याख्यामा हामी जिम्मेवार हुँदैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
